# MINT-TTS — Egyptian Arabic homograph experiment

**The question.** Egyptian Arabic is written without short vowels, so the same
spelling carries several pronunciations and only context decides which:

| written | reading A | reading B |
|---|---|---|
| `علم` | `عَلَم` *3alam* — flag | `عِلْم` *3elm* — science |
| `عمرك` | `عُمرَك` *3omrak* — to a man | `عُمرِك` *3omrik* — to a woman |
| `ضرب` | `ضَرَب` — he hit | `ضُرِب` — he was hit |

Most Arabic TTS systems sidestep this by requiring diacritised input, which
moves the problem to whoever types the text. This experiment asks whether the
model can resolve it from context alone, and whether it spends **more
computation** on the words that need it.

**Three components, and what each one does:**

1. **`ar_char` frontend** — raw graphemes, so the ambiguity actually reaches
   the model. (On English, the `ipa` frontend pre-resolved every homograph and
   the experiment measured nothing. That null result is why this matters.)
2. **Frozen MARBERTv2** — one contextual vector per word. Character statistics
   over 68 hours cannot recover lexical semantics; a dialect-pretrained LM
   already has them. Cached offline, so training never runs BERT.
3. **Difficulty-aware ACT routing** — an ambiguous token pays 25% of the
   per-step compute price, so depth where it is needed is affordable while
   easy words stay under pressure.

**Runtime.** Sized for an H200. Preprocessing is the slow part (~40–70 min for
100 h); training runs at roughly 4–6 it/s at batch 64.

---
## 0. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# The repo. Skip the clone if you already have it mounted.
import os, sys
from pathlib import Path

REPO = Path("MINT-TTS")
if not REPO.exists():
    !git clone https://github.com/MohammedAly22/MINT-TTS.git
os.chdir(REPO if REPO.exists() else ".")
sys.path.insert(0, str(Path.cwd()))
print("cwd:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt
# Arabic-specific: transformers for MARBERT, datasets/soundfile/librosa for the corpus.
!pip install -q transformers datasets soundfile librosa

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

---
## 1. What the frontend does to Arabic

Before any training, check that the text pipeline preserves what the model
needs. The undiacritised spellings must survive intact — if the normaliser
stripped or folded them, the experiment would be impossible and the failure
would be invisible later.

In [ ]:
from mint_tts.text.arabic import ArabicNormalizer, ARABIC_WORD_RE

norm = ArabicNormalizer()
examples = [
    "انا رسمت علم مصر",                       # 3alam = flag
    "انا بحب العلم جدا و نفسي ابقا عالم",      # 3elm  = science
    "عمرك فكرتي الراجل بتاع غزل البنات؟",      # 3omrik (feminine addressee)
    "عندي ٢٥ كتاب و ٥٠٪ منهم عربي",           # Arabic-Indic digits + percent
    "قال ليـــ إزيك يا آدم؟",                  # tatweel, hamza, Arabic '?'
    "عَلَم مُشَكَّل",                            # diacritics get stripped
]
for raw in examples:
    out = norm(raw)
    print(f"in  : {raw}")
    print(f"out : {out}")
    print(f"words: {ARABIC_WORD_RE.findall(out)}\n")

### The difficulty signal

Every word gets a score that tells the compute penalty where to relax. This is
a *prior*, not a target — the router is never trained to reproduce it, and the
model never sees it at inference time.

In [ ]:
from mint_tts.text.homographs_ar import difficulty_profile, sentence_difficulty

for raw in ["عامل ايه النهاردة؟",
            "انا رسمت علم مصر",
            "عمرك فكرتي الراجل بتاع غزل البنات بينفخ الكيس ازاي؟ هسيبك تجاوبي و تخمني"]:
    words = ARABIC_WORD_RE.findall(norm(raw))
    prof = difficulty_profile(words)
    print(f"sentence difficulty {sentence_difficulty(words):.2f}")
    print("  " + "  ".join(f"{w}={d:.1f}" for w, d in zip(words, prof)) + "\n")

The easy greeting scores **0.00** — it should be generated fast and shallow.
The last sentence flags `عمرك` and `هسيبك`, whose vowels depend on the
feminine verb `فكرتي` several words away. That distance is the point: a
fixed-depth encoder has a fixed number of hops to find that evidence.

### Does MARBERT actually separate the readings?

This is the load-bearing assumption of the whole design. If the LM gives `علم`
the same vector in both sentences, nothing downstream can disambiguate it and
the architecture is pointless. Check before training, not after.

In [ ]:
from mint_tts.modules.semantic import SemanticEncoder
import numpy as np

enc = SemanticEncoder("marbert", device="cuda" if torch.cuda.is_available() else "cpu")

def vec(sentence, target):
    words = ARABIC_WORD_RE.findall(norm(sentence))
    f = enc.encode(words)
    return f.vectors[words.index(target)]

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

flag    = vec("انا رسمت علم مصر", "علم")
science = vec("انا بحب العلم جدا و نفسي ابقا عالم لما اكبر", "العلم")
control = vec("انا رسمت علم السعودية", "علم")   # same reading, different sentence

print(f"flag vs science  : {cos(flag, science):.3f}   <- should be LOWER")
print(f"flag vs flag     : {cos(flag, control):.3f}   <- should be HIGHER")
print()
print("If the first number is not clearly below the second, the LM is not")
print("separating the readings and the semantic path cannot help. Try")
print("`layer=-5` (often more lexical) or a different model before training.")

---
## 2. The corpus

15,653 clips, one speaker, 24 kHz, undiacritised and unpunctuated. The
official splits are disjoint **by source video**, and `prepare_egyptian.py`
honours them — a random split would put near-duplicates of training clips into
validation and make every validation number optimistic.

In [ ]:
# ~12 GB download + wav export. Add --limit 200 for a quick dry run first.
!python scripts/prepare_egyptian.py --out data/egyptian --min-confidence 0.0

The script ends with a **homograph coverage** report. Read it before going
further: if almost no utterance contains an ambiguous word, the corpus cannot
teach disambiguation and a null result would say nothing about the
architecture. Anything above a few percent of utterances is workable.

---
## 3. Preprocessing

Mel/pitch/energy, cached tokenisation, **and** the frozen MARBERT vectors.
Caching the LM output here is what keeps training fast: a 163M-parameter BERT
forward pass per step would otherwise dominate a ~18M-parameter acoustic
model.

The vectors are stored under a hash of `(model, layer)`, so switching LM later
cannot silently reuse the old model's features.

In [ ]:
!python scripts/preprocess.py --config configs/egyptian_homograph.yaml --workers 8

In [ ]:
import json
stats = json.loads(Path("data/preprocessed/egyptian/stats.json").read_text(encoding="utf-8"))
for k in ["n_utterances", "total_hours", "vocab_size", "input_type",
          "semantic_model", "semantic_hidden_size", "semantic_files"]:
    print(f"{k:22s} {stats.get(k)}")

---
## 4. A real vocoder

Griffin-Lim is the zero-download default and sounds rough — rough enough that
you cannot judge a homograph by ear through it. Fetch a 24 kHz neural vocoder
before listening to anything.

The vocoder is **frozen and shared** across every run, so a quality difference
between two experiments can never come from the vocoder.

In [ ]:
# A 24 kHz universal HiFi-GAN. Any 24 kHz vocoder works; the sample rate
# must match configs/dataset_egyptian.yaml (24000).
!python scripts/download_vocoder.py --hf-repo nvidia/tts_hifigan --hf-file '*.ckpt' || \
 echo "Fetch failed -- training still works, but audio will be Griffin-Lim.

---
## 5. Train

Watch these, in this order. Each one can invalidate everything below it:

| metric | meaning | act if |
|---|---|---|
| `align/entropy_ratio` | aligner health | still > 0.5 at 8k steps → stop, nothing downstream is meaningful |
| `val/mcd_vs_chance` | is the audio utterance-specific? | ≥ 1.0 → the model says the same thing regardless of input |
| `semantic/delta_norm` | is the LM path being used? | stays 0 → semantics are dead, result would be from elsewhere |
| `compute/difficulty_contrast` | **the claim** | should rise above 0 after the warmup |
| `compute/encoder_depth_spread` | is the router differentiating? | ~0 → collapsed to a constant |
| `homograph/divergence_ratio` | do readings differ? | ~1 → same pronunciation in both contexts |
| `probe/length_corr` | the trivial solution | ~1 → router only learned sentence length |

Compute pressure does not start until step 12,000 (`loss.compute.warmup_steps`).
Before that the router is held at full depth on purpose: an unpenalised router
collapses to a constant and takes the alignment down with it.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
!python scripts/train.py --config configs/egyptian_homograph.yaml

### Resuming

Checkpoints land in `runs/egyptian_homograph/checkpoints/`. To continue after
a disconnect:

```bash
python scripts/train.py --config configs/egyptian_homograph.yaml \
    --resume runs/egyptian_homograph/checkpoints/final.pt
```

---
## 6. Listen

The automatic metric says whether two renderings **differ**. It cannot say
whether they differ *correctly* — only a speaker can. Synthesise both readings
and listen.

In [ ]:
from mint_tts.inference.synthesize import Synthesizer
from IPython.display import Audio, display

syn = Synthesizer.from_checkpoint("runs/egyptian_homograph/checkpoints/best.pt")

pairs = [
    ("علم — flag",    "انا رسمت علم مصر"),
    ("علم — science", "انا بحب العلم جدا و نفسي ابقا عالم لما اكبر"),
    ("عمرك — to a man",   "يا عم عمرك شفت حاجة زي كده يا راجل"),
    ("عمرك — to a woman", "عمرك فكرتي الراجل بتاع غزل البنات بينفخ الكيس ازاي"),
]
for label, text in pairs:
    res = syn(text, quality=0.9)
    print(f"--- {label}")
    print(res.summary())
    display(Audio(res.wav.cpu().numpy(), rate=res.sample_rate))

### Where did the compute go?

Per-word depth for one sentence. If the hypothesis holds, the ambiguous words
sit clearly above the rest — and the easy function words sit at the floor.

In [ ]:
import matplotlib.pyplot as plt

text = "عمرك فكرتي الراجل بتاع غزل البنات بينفخ الكيس ازاي"
res = syn(text, quality=0.9)

words = res.encoded.words
depth = res.word_complexity
diff = difficulty_profile(words)
colors = ["#d9534f" if d > 0.5 else "#5b8def" for d in diff]

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(range(len(words)), depth, color=colors)
ax.set_xticks(range(len(words)))
# Reshape the Arabic labels so matplotlib renders them correctly.
try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    labels = [get_display(arabic_reshaper.reshape(w)) for w in words]
except ImportError:
    labels = words
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=12)
ax.set_ylabel("encoder depth (fraction of max)")
ax.set_title("red = flagged ambiguous")
ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

for w, d, f in zip(words, depth, diff):
    print(f"  {w:>12s}  depth={d:.3f}  difficulty={f:.1f}")

### The speed claim

The easy sentence should be both cheaper and faster than the hard one. If they
cost the same, the router is not allocating.

In [ ]:
import time

tests = [
    ("easy", "عامل ايه النهاردة؟"),
    ("long but easy", "انا رحت السوق و اشتريت عيش و لبن و جبنة و زيتون و رجعت البيت"),
    ("hard (homographs)", "عمرك فكرتي الراجل بتاع غزل البنات بينفخ الكيس ازاي؟ هسيبك تجاوبي و تخمني"),
]
print(f"{'sentence':<22s} {'words':>6s} {'depth':>7s} {'RTF':>8s} {'saving':>8s}")
for label, text in tests:
    r = syn(text, quality=0.9)                      # warm up
    t0 = time.perf_counter()
    r = syn(text, quality=0.9)
    dt = time.perf_counter() - t0
    audio_s = r.mel.shape[-1] * syn.cfg.audio.hop_length / syn.cfg.audio.sample_rate
    print(f"{label:<22s} {len(r.encoded.words):>6d} "
          f"{r.token_complexity.mean():>7.3f} {dt/max(audio_s,1e-6):>8.4f} "
          f"{r.flops.saving*100:>7.1f}%")

### The quality/compute curve

One checkpoint spans the whole trade-off, because `q` was sampled during
training. Low `q` should be cheap and rough; high `q` deeper and better.

In [ ]:
text = "انا بحب العلم جدا و نفسي ابقا عالم لما اكبر"
for q in [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    r = syn(text, quality=q)
    print(f"q={q:<4.1f} depth={r.token_complexity.mean():.3f} "
          f"FLOPs={r.flops.total:.2e} saving={r.flops.saving*100:5.1f}%")
    display(Audio(r.wav.cpu().numpy(), rate=r.sample_rate))

---
## 7. The controls

A positive result on the main run means little on its own. These two ablations
are what make it interpretable, and they are cheap — preprocessing is shared,
so only training re-runs.

**`egyptian_nosemantic`** — identical except MARBERT is off. If this also
separates the readings, the character encoder was sufficient and the LM is
dead weight. If it flattens, the semantic path is doing the work.

**`egyptian_dense`** — no routing at all. Gives the quality ceiling, and shows
whether the disambiguation needs adaptive depth or just the LM features.

In [ ]:
!python scripts/train.py --config configs/egyptian_nosemantic.yaml
# !python scripts/train.py --config configs/egyptian_dense.yaml

In [ ]:
# Compare the runs on the numbers that matter.
from tensorboard.backend.event_processing import event_accumulator

def final(run, tag):
    import glob
    files = glob.glob(f"runs/{run}/**/events.out.tfevents.*", recursive=True)
    if not files:
        return None
    ea = event_accumulator.EventAccumulator(max(files))
    ea.Reload()
    if tag not in ea.Tags().get("scalars", []):
        return None
    return ea.Scalars(tag)[-1].value

TAGS = ["homograph/divergence_ratio", "compute/difficulty_contrast",
        "probe/contrast", "probe/length_corr", "val/mcd", "semantic/delta_norm"]
runs = ["egyptian_homograph", "egyptian_nosemantic", "egyptian_dense"]

print(f"{'metric':<34s}" + "".join(f"{r[9:]:>16s}" for r in runs))
for tag in TAGS:
    row = "".join(
        f"{v:>16.4f}" if (v := final(r, tag)) is not None else f"{'-':>16s}"
        for r in runs)
    print(f"{tag:<34s}{row}")

---
## 8. How to read the outcome

**The result is positive if,** on `egyptian_homograph`:

- `homograph/divergence_ratio` is comfortably above 1 (the readings differ
  more than ordinary contextual variation), **and**
- `compute/difficulty_contrast` is above 0 (ambiguous tokens get more depth),
  **and**
- `probe/length_corr` is *not* near 1 (it is not just sentence length), **and**
- the `egyptian_nosemantic` control is clearly weaker on the first two, **and**
- the tongue twisters stayed cheap — hard phonetics with easy semantics should
  *not* light up, or "difficulty" means something other than ambiguity.

**And then you still listen.** Differing is not the same as differing
correctly. Every automatic number here is necessary and none is sufficient;
the audio from section 6 is the actual evidence.

**If `semantic/delta_norm` stayed at 0**, the semantic path never escaped its
zero initialisation and every homograph number is coming from somewhere else.
Check that preprocessing wrote the vectors and that the hidden size matches.

**If the tongue twisters cost as much as the homographs**, the router is
tracking something phonetic or character-level rather than ambiguity. That is
a real and publishable finding, but it is a different claim than the one this
experiment set out to test — report it as what it is.